# Side Channel Attack 실습

> 강의 슬라이드: *보안 프로그래밍 — Side Channel 공격과 방어*

본 노트북에서는 슬라이드에서 다룬 부채널 공격 중 **세 가지를 직접 실행**해봅니다.

| # | 공격 | 카테고리 | 핵심 메시지 |
|---|------|---------|------------|
| 1 | **Bellcore Attack** (RSA-CRT) | Active / Fault | 단 1번의 fault만으로 RSA 개인키 전체 복원 |
| 2 | **Timing Attack** (Password compare) | Passive / Time | 조기 종료 비교의 시간 차로 비밀값 한 글자씩 복원 |
| 3 | **SPA** (Square-and-Multiply) | Passive / Power | 전력 파형의 패턴만으로 비밀 지수 복원 |

각 공격마다 **방어 코드**도 함께 실행하여 *왜* 그것이 방어가 되는지 시각적으로 확인합니다.

> Google Colab에서 바로 실행 가능합니다. `Runtime → Run all`을 사용하세요.


In [ ]:
# 필수 라이브러리 설치 (Colab에는 sympy가 기본 설치되어 있지만 안전을 위해 명시)
!pip install -q sympy numpy matplotlib
print("환경 준비 완료")

---
# 1. Bellcore Attack on RSA-CRT

## 이론

RSA-CRT는 RSA 서명/복호화를 **약 4배 빠르게** 만드는 표준 최적화입니다. 거의 모든 스마트카드·HSM·OpenSSL이 사용합니다. 하지만 1997년 Boneh, DeMillo, Lipton(벨코어 연구소)이 발견한 것은: **단 1번의 연산 오류만 있으면 개인키 전체가 복원된다**는 사실입니다.

**RSA-CRT 서명 과정:**

$$s_p = m^{d_p} \bmod p, \quad s_q = m^{d_q} \bmod q$$

$$s = \text{CRT}(s_p, s_q) = s_q + q \cdot (q^{-1}(s_p - s_q) \bmod p)$$

**공격 원리:**

$s_p$ 계산 도중 단 한 비트라도 결함이 발생하여 faulty signature $s'$이 나오면:

- $s' \equiv s \pmod{q}$ 는 여전히 성립 ($s_q$ 쪽은 정상)
- $s' \not\equiv s \pmod{p}$ ($s_p$ 쪽이 망가짐)

따라서 검증식 $(s')^e \bmod n$을 계산하면 $q$의 배수만큼 $m$과 차이가 납니다:

$$\gcd\left((s')^e - m,\ n\right) = q$$

→ $n$을 $p \times q$로 인수분해 완료, 개인키 $d$ 복원.

**실용적 의미:** Laser fault injection이나 voltage glitching으로 단 한 번 fault를 주입할 수 있으면 RSA-2048도 무력화됩니다. CC 평가에서 AVA_VAN.5에 fault injection 저항성이 명시된 이유입니다.


In [ ]:
from sympy import randprime, mod_inverse
from math import gcd
import random

random.seed(2026)

def generate_rsa_crt_key(bits: int = 512) -> dict:
    '''RSA-CRT 키 생성. 강의용으로 512-bit를 사용 (실제는 2048+)'''
    while True:
        p = randprime(2**(bits-1), 2**bits)
        q = randprime(2**(bits-1), 2**bits)
        if p != q:
            break
    n = p * q
    e = 65537
    phi = (p - 1) * (q - 1)
    d = mod_inverse(e, phi)
    return {
        'n': n, 'e': e, 'd': d,
        'p': p, 'q': q,
        'dp': d % (p - 1),
        'dq': d % (q - 1),
        'qinv': mod_inverse(q, p),
    }

key = generate_rsa_crt_key(bits=512)
print(f"n (modulus, 공개) = {hex(key['n'])[:50]}...")
print(f"p (비밀 인수)     = {hex(key['p'])[:50]}...")
print(f"q (비밀 인수)     = {hex(key['q'])[:50]}...")
print(f"d (개인키)        = {hex(key['d'])[:50]}...")

In [ ]:
def rsa_crt_sign(m: int, key: dict, induce_fault: bool = False,
                 fault_bit: int = None) -> int:
    '''RSA-CRT 서명. induce_fault=True 이면 s_p 계산 직후 한 비트를 뒤집음'''
    sp = pow(m, key['dp'], key['p'])
    sq = pow(m, key['dq'], key['q'])

    if induce_fault:
        # ★ Fault injection 모델링: 단일 bit-flip
        # 실제 공격에서는 voltage glitch / laser pulse 등으로 발생
        if fault_bit is None:
            fault_bit = random.randint(0, 200)
        sp = sp ^ (1 << fault_bit)

    # CRT recombination (Garner's formula)
    h = (key['qinv'] * (sp - sq)) % key['p']
    s = sq + h * key['q']
    return s


# 메시지(또는 메시지의 해시값)에 서명
m = 0xDEADBEEFCAFEBABE1234567890

s_correct = rsa_crt_sign(m, key)
s_faulty  = rsa_crt_sign(m, key, induce_fault=True)

# 두 서명을 RSA 검증
verify_ok      = pow(s_correct, key['e'], key['n']) == m
verify_faulty  = pow(s_faulty,  key['e'], key['n']) == m

print(f"정상 서명 검증 결과:    {verify_ok}")
print(f"Faulty 서명 검증 결과:  {verify_faulty}")
print(f"\\n→ Faulty 서명은 검증에 실패하지만,")
print(f"   공격자에게는 이 한 개의 잘못된 서명만 있으면 충분합니다.")

In [ ]:
def bellcore_attack(m: int, s_faulty: int, n: int, e: int):
    '''
    Bellcore 공격: 단 하나의 faulty 서명으로 n을 인수분해.

    공격자에게 필요한 것:
      - 공개 정보: m, n, e
      - 단 하나의 faulty 서명: s_faulty
    '''
    diff = (pow(s_faulty, e, n) - m) % n
    factor = gcd(diff, n)
    if 1 < factor < n:
        return factor, n // factor
    return None, None


# ── 공격 실행 ─────────────────────────────────────────
f1, f2 = bellcore_attack(m, s_faulty, key['n'], key['e'])

print("=" * 60)
print("[ Bellcore Attack 결과 ]")
print("=" * 60)
print(f"복원된 인수 1: {hex(f1)[:60]}")
print(f"복원된 인수 2: {hex(f2)[:60]}")
print()
print(f"실제 p, q:    {hex(key['p'])[:60]}")
print(f"              {hex(key['q'])[:60]}")
print()
print(f"인수분해 성공: {{f1, f2}} == {{key['p'], key['q']}}")

# 비밀키 d까지 복원
phi_rec = (f1 - 1) * (f2 - 1)
d_rec = mod_inverse(key['e'], phi_rec)
print(f"\\n복원된 개인키 d == 실제 d: {d_rec == key['d']}")
print("\\n→ 공격자가 모든 향후 서명/복호화를 자유롭게 수행 가능")

### Bellcore 공격 방어

표준 방어는 **서명 후 검증 (Sign-then-Verify)** 입니다:

```python
s = crt_sign(m, key)
if pow(s, e, n) != m:
    # 결함 감지! 출력하지 않음
    raise FaultDetected
return s
```

서명이 잘못되면 출력하지 않으므로 공격자가 faulty signature를 손에 넣지 못합니다.
하지만 이 검증 자체에도 fault를 주입하는 **이중 fault 공격**이 존재하므로, 실제 인증 제품에서는 redundant computation + integrity check 조합을 사용합니다.


In [ ]:
def rsa_crt_sign_safe(m: int, key: dict, induce_fault: bool = False):
    '''Sign-then-Verify로 Bellcore 공격 방어'''
    s = rsa_crt_sign(m, key, induce_fault=induce_fault)
    # ★ 핵심 방어: 출력 전 검증
    if pow(s, key['e'], key['n']) != m:
        raise ValueError("Fault detected — signature not released")
    return s

# 1) 정상 서명
try:
    s_ok = rsa_crt_sign_safe(m, key, induce_fault=False)
    print(f"정상 서명 출력: {hex(s_ok)[:50]}...")
except ValueError as e:
    print(f"오류: {e}")

# 2) Fault 주입 시도
try:
    s_bad = rsa_crt_sign_safe(m, key, induce_fault=True)
    print(f"Fault 서명 출력: {hex(s_bad)[:50]}...")
except ValueError as e:
    print(f"\\n방어 성공 → {e}")
    print("→ 공격자는 faulty signature 자체를 얻지 못함")

---
# 2. Timing Attack on Password / MAC Comparison

## 이론

문자열을 한 글자씩 비교하면서 **첫 mismatch에서 조기 종료**하는 코드는, 공격자가 측정한 응답 시간으로 *몇 개 글자가 일치했는지*를 알 수 있게 합니다.

```python
def insecure_compare(a, b):
    for i in range(len(a)):
        if a[i] != b[i]:
            return False        # 조기 종료 — timing leak의 원인
    return True
```

| 입력 | 일치 글자 수 | 소요 시간 |
|------|------------|----------|
| `Xxxxxxx` | 0 | 가장 빠름 |
| `Sxxxxxx` | 1 | 조금 더 |
| `S3xxxxx` | 2 | 더 |
| ... | ... | ... |

각 위치에서 **시간이 가장 오래 걸리는 글자**가 정답입니다. 검색 공간이 $|\Sigma|^n$ 에서 $|\Sigma| \cdot n$ 으로 줄어듭니다.

**실제 사례:**
- 2009 Google Keyczar — HMAC 비교 취약점
- 2013 Lucky13 (CVE-2013-0169) — TLS CBC MAC-then-Encrypt
- 2003 Boneh & Brumley — OpenSSL RSA 원격 타이밍 공격

> Python 인터프리터는 시스템 노이즈가 크기 때문에, 측정 차이를 명확히 보이도록 본 데모는 비교 후 의도적으로 짧은 작업을 추가했습니다. 실제 C/하드웨어에서는 이런 증폭 없이도 측정 가능합니다 (수백~수천 회 반복 측정 필요).


In [ ]:
import time
import statistics
import string
import hmac
import hashlib

# 공격 대상의 비밀 패스워드
SECRET = "S3cret!"

def insecure_compare(guess: str, secret: str) -> bool:
    '''취약: 일치하는 글자가 많을수록 더 많은 일을 함 → 시간 차이 발생'''
    if len(guess) != len(secret):
        return False
    for i in range(len(guess)):
        if guess[i] != secret[i]:
            return False
        # 실제 시스템의 per-byte 메모리/캐시 작업을 모델링.
        # Python 자체 비교는 너무 빨라 측정이 어렵기 때문에
        # 큰 페이로드(64KB)에 대한 SHA-256으로 시간을 증폭한다.
        # 한 번에 ~150 마이크로초가 걸려 OS 인터럽트 노이즈(~수십 us)보다 훨씬 크다.
        # (실제 C 구현·HW에서는 이런 증폭 없이도 측정 가능)
        hashlib.sha256(b'x' * 65536).digest()
    return True

def measure(guess: str, secret: str, trials: int = 50) -> float:
    '''
    노이즈 제거 전략:
      - perf_counter_ns로 trials회 측정
      - 정렬 후 하위 10%의 평균만 사용
      - OS 인터럽트/GC는 시간을 '더하기'만 하므로,
        진짜 시그널은 측정값의 하단(최소값 근처)에 있다
    '''
    times = []
    for _ in range(trials):
        s = time.perf_counter_ns()
        insecure_compare(guess, secret)
        e = time.perf_counter_ns()
        times.append(e - s)
    times.sort()
    return statistics.mean(times[:max(3, trials // 10)])

# Sanity check
t_wrong = measure("X" * len(SECRET), SECRET)
t_right = measure(SECRET, SECRET)
print(f"전부 틀린 입력의 측정 시간: {t_wrong:>8.0f} ns")
print(f"전부 맞는 입력의 측정 시간: {t_right:>8.0f} ns")
print(f"차이: {t_right - t_wrong:.0f} ns  →  측정 가능한 leak 확인")
print(f"한 글자가 일치할 때마다 약 {(t_right - t_wrong) / len(SECRET):.0f} ns 증가")

In [ ]:
def recover_password_by_timing(secret: str, alphabet: str = None) -> str:
    '''
    각 위치에서 가장 오래 걸리는 글자를 선택.
    검색 공간: |alphabet|^len(secret)  →  |alphabet| * len(secret)
    '''
    if alphabet is None:
        alphabet = string.ascii_letters + string.digits + "!@#$%^&*"

    target_len = len(secret)
    recovered = ""

    print(f"검색 공간 비교:")
    print(f"  Brute force:    {len(alphabet)**target_len:>20,} 시도")
    print(f"  Timing attack:  {len(alphabet)*target_len:>20,} 시도\\n")
    print(f"한 글자씩 복원 시작 (정답: {repr(secret)}):\\n")

    for pos in range(target_len):
        times = {}
        for c in alphabet:
            guess = recovered + c + "X" * (target_len - pos - 1)
            times[c] = measure(guess, secret)

        best = max(times, key=times.get)
        recovered += best

        # 상위 3개를 함께 출력
        top3 = sorted(times.items(), key=lambda x: -x[1])[:3]
        top3_str = ", ".join(f"{repr(c)}:{t:.0f}" for c, t in top3)
        status = "OK" if best == secret[pos] else "FAIL"
        print(f"  pos={pos} -> {repr(best)} [{status}]  | top3: {top3_str}")

    return recovered

recovered = recover_password_by_timing(SECRET)
print(f"\\n복원 결과: {repr(recovered)}")
print(f"실제 값:   {repr(SECRET)}")
print(f"성공: {recovered == SECRET}")

### 시각화: 시간 차이로 정답이 도드라지는 모습

첫 글자를 추측할 때, **올바른 문자만 명확히 더 오래 걸린다**는 것을 그래프로 확인합니다.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

alphabet = string.ascii_uppercase + string.ascii_lowercase + string.digits + "!@#$%"
times_per_char = {}
for c in alphabet:
    guess = c + "X" * (len(SECRET) - 1)
    times_per_char[c] = measure(guess, SECRET)

chars = list(times_per_char.keys())
values = [times_per_char[c] for c in chars]

colors = ['#d62728' if c == SECRET[0] else '#1f77b4' for c in chars]
plt.figure(figsize=(14, 4))
plt.bar(range(len(chars)), values, color=colors)
plt.xticks(range(len(chars)), chars, fontsize=7, rotation=90)
plt.ylabel("Time, lower-quartile mean (ns)")
plt.title(f"Timing attack - first character (correct = '{SECRET[0]}' in red)")
plt.axhline(np.median(values), color='gray', linestyle='--', alpha=0.5, label='Median')
plt.legend()
plt.tight_layout()
plt.show()

print(f"올바른 글자 '{SECRET[0]}' 의 시간: {times_per_char[SECRET[0]]:.0f} ns")
print(f"전체 평균 시간:             {np.mean(values):.0f} ns")
print(f"올바른 글자의 z-score:      {(times_per_char[SECRET[0]] - np.mean(values)) / np.std(values):.2f}")

### 방어: 상수 시간 비교

`hmac.compare_digest()`는 길이가 같다면 **항상 끝까지 비교**하고, 결과를 누적 XOR로 모읍니다.
조건 분기 없이 데이터-독립적 시간을 보장합니다.


In [ ]:
def secure_compare(guess: str, secret: str) -> bool:
    '''hmac.compare_digest: 상수 시간 비교 (Python 표준 라이브러리)'''
    return hmac.compare_digest(guess, secret)

def measure_secure(guess: str, secret: str, trials: int = 50) -> float:
    times = []
    for _ in range(trials):
        s = time.perf_counter_ns()
        secure_compare(guess, secret)
        e = time.perf_counter_ns()
        times.append(e - s)
    times.sort()
    return statistics.mean(times[:max(3, trials // 10)])

# 측정: 0~7글자 일치하는 입력의 시간 (insecure vs secure)
match_counts = list(range(len(SECRET) + 1))
insecure_means = []
secure_means = []

for k in match_counts:
    guess = SECRET[:k] + "X" * (len(SECRET) - k)
    insecure_means.append(measure(guess, SECRET))
    secure_means.append(measure_secure(guess, SECRET))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(match_counts, insecure_means, 'o-', color='#d62728', linewidth=2)
axes[0].set_title("insecure_compare - time grows with matching prefix length")
axes[0].set_xlabel("Number of leading matching characters")
axes[0].set_ylabel("Time (ns)")
axes[0].grid(alpha=0.3)

axes[1].plot(match_counts, secure_means, 'o-', color='#2ca02c', linewidth=2)
axes[1].set_title("hmac.compare_digest - flat (constant time)")
axes[1].set_xlabel("Number of leading matching characters")
axes[1].set_ylabel("Time (ns)")
axes[1].grid(alpha=0.3)

# 같은 y-axis 스케일로 비교
ymax = max(max(insecure_means), max(secure_means)) * 1.1
axes[0].set_ylim(0, ymax)
axes[1].set_ylim(0, ymax)

plt.tight_layout()
plt.show()

print("\\n→ Insecure 버전: 일치 글자 수가 곧 시간 = leak")
print("→ Secure 버전:   기울기 ≈ 0 = no leak")

---
# 3. Simple Power Analysis (SPA) on RSA Square-and-Multiply

## 이론

Square-and-Multiply (left-to-right):
```python
result = 1
for bit in bin(d)[2:]:        # MSB → LSB
    result = result * result mod n      # ① square: 매 비트 발생
    if bit == 1:
        result = result * base mod n    # ② multiply: bit=1일 때만 발생
```

전력 소비:
- **Square** 연산: 매 비트마다 발생 (이 데모에서 평균 전력 0.3)
- **Multiply** 연산: 비트가 1일 때만 추가 (평균 전력 0.7)

→ 단 **하나의 power trace**만 봐도 패턴으로 비밀 지수 $d$를 그대로 읽어낼 수 있음.

이것이 RSA / DH / ECC scalar multiplication에서 **always-execute** 방식 (Montgomery Ladder, double-and-add-always)을 쓰는 이유입니다.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

SQUARE_SAMPLES = 20      # 한 번의 square가 차지하는 sample 수
MULTIPLY_SAMPLES = 30    # 한 번의 multiply가 차지하는 sample 수
POWER_SQUARE = 0.3       # square 평균 전력
POWER_MULTIPLY = 0.7     # multiply 평균 전력

def simulate_power_trace(exponent: int, noise: float = 0.05):
    '''비밀 지수로 square-and-multiply 실행 시의 전력 trace를 시뮬레이션'''
    bits = bin(exponent)[2:]   # MSB-first
    trace = []
    for bit in bits:
        # 매 비트: square 발생
        trace.extend(POWER_SQUARE + np.random.normal(0, noise, SQUARE_SAMPLES))
        if bit == '1':
            # bit=1: multiply 추가 발생
            trace.extend(POWER_MULTIPLY + np.random.normal(0, noise, MULTIPLY_SAMPLES))
    return np.array(trace), bits

# 비밀 지수
secret_d = 0b10110101   # = 181
trace, true_bits = simulate_power_trace(secret_d)

# Trace 시각화
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(trace, linewidth=0.8, color='#1f77b4')
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='threshold')
ax.set_xlabel("Sample")
ax.set_ylabel("Power")
ax.set_title(f"Power trace — secret d = 0b{true_bits} (= {secret_d})")
ax.legend()
ax.grid(alpha=0.3)

# 각 비트가 시작되는 위치에 비트 값을 라벨로 표시
pos = 0
for bit in true_bits:
    ax.text(pos + SQUARE_SAMPLES // 2, 1.0, bit,
            ha='center', fontsize=11, fontweight='bold',
            color='green' if bit == '1' else 'orange')
    pos += SQUARE_SAMPLES
    if bit == '1':
        pos += MULTIPLY_SAMPLES

ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.show()

print(f"\\n위 trace에서 비밀 지수 d = 0b{true_bits} 의 패턴이 육안으로 보입니다.")
print("• 낮은 peak만 있으면 → bit 0")
print("• 낮은 peak 다음에 높은 peak → bit 1")

In [ ]:
def recover_exponent_from_trace(trace: np.ndarray,
                                threshold: float = 0.5,
                                square_size: int = SQUARE_SAMPLES) -> str:
    '''
    Power trace에서 비트 복원.
    핵심 아이디어:
      - Low 구간 = 연속된 square들 (길이/SQUARE_SIZE 만큼의 0-비트 다음 마지막에 1-비트가 올 수 있음)
      - High 구간 = 단일 multiply (마지막 square와 함께 하나의 1-비트를 구성)
    '''
    high = trace > threshold

    # 연속 구간 (state, length) 추출
    segments = []
    cur, length = high[0], 1
    for v in high[1:]:
        if v == cur:
            length += 1
        else:
            segments.append((cur, length))
            cur, length = v, 1
    segments.append((cur, length))

    # 비트 복원
    bits = ""
    i = 0
    while i < len(segments):
        state, length = segments[i]
        if not state:  # Low (squares)
            num_squares = max(1, round(length / square_size))
            if i + 1 < len(segments) and segments[i+1][0]:
                # 마지막 square가 다음 multiply와 짝지어짐 → 그 비트는 1, 앞쪽은 모두 0
                bits += '0' * (num_squares - 1) + '1'
                i += 2
            else:
                # 뒤에 multiply 없음 → 전부 lone square (= 0-비트들)
                bits += '0' * num_squares
                i += 1
        else:
            i += 1
    return bits


# 다양한 비밀 지수로 검증
print("[ SPA 복원 결과 ]\\n")
print(f"{'secret d':>5} | {'true bits':>10} | {'recovered':>10} | match")
print("-" * 50)
for d in [0b10110101, 0b11001011, 0b10000001, 0b11111111, 0b10101010, 0b11100100]:
    trace, true_bits = simulate_power_trace(d)
    rec = recover_exponent_from_trace(trace)
    match = '✓' if rec == true_bits else '✗'
    print(f"{d:>5} | {true_bits:>10} | {rec:>10} |  {match}")

### 방어: Montgomery Ladder

각 비트마다 **항상 동일한 횟수의 square + multiply를 수행**하여 전력 패턴을 균일화합니다.
연산량은 약 2배지만 부채널 leak이 사라집니다.

```python
def montgomery_ladder(base, exp, n):
    r0, r1 = 1, base
    for bit in bin(exp)[2:]:
        if bit == 0:
            r1 = (r0 * r1) % n   # multiply
            r0 = (r0 * r0) % n   # square
        else:
            r0 = (r0 * r1) % n   # multiply
            r1 = (r1 * r1) % n   # square
    return r0
```

각 분기에서 정확히 **1 square + 1 multiply**가 발생합니다.


In [ ]:
def montgomery_ladder(base: int, exp: int, modulus: int) -> int:
    '''상수 패턴 모듈러 지수승 — SPA 저항성'''
    r0, r1 = 1, base
    for bit in bin(exp)[2:]:
        if bit == '0':
            r1 = (r0 * r1) % modulus
            r0 = (r0 * r0) % modulus
        else:
            r0 = (r0 * r1) % modulus
            r1 = (r1 * r1) % modulus
    return r0

# 일반 pow()와 동일한 결과인지 확인
test_n = 65537
for _ in range(5):
    base = random.randint(2, test_n - 1)
    exp  = random.randint(2, 10000)
    ref = pow(base, exp, test_n)
    mlr = montgomery_ladder(base, exp, test_n)
    assert ref == mlr, f"불일치! {base}^{exp} mod {test_n}"
print("Montgomery Ladder의 출력이 일반 pow()와 모두 일치 ✓")

In [ ]:
# Montgomery Ladder의 전력 trace — 매 비트마다 동일한 패턴
def simulate_montgomery_trace(exponent: int, noise: float = 0.05):
    bits = bin(exponent)[2:]
    trace = []
    SAMPLES = 25
    for _ in bits:
        # 매 비트마다 1 square + 1 multiply (순서는 분기에 따라 다르지만 전력 패턴은 동일)
        trace.extend(POWER_SQUARE    + np.random.normal(0, noise, SAMPLES))
        trace.extend(POWER_MULTIPLY  + np.random.normal(0, noise, SAMPLES))
    return np.array(trace)

trace_naive, _ = simulate_power_trace(secret_d)
trace_ladder    = simulate_montgomery_trace(secret_d)

fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharey=True)

axes[0].plot(trace_naive, color='#d62728', linewidth=0.8)
axes[0].set_title(f"[BAD] Naive Square-and-Multiply - bit pattern {true_bits} is visible")
axes[0].set_ylabel("Power")
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.4)
axes[0].grid(alpha=0.3)

axes[1].plot(trace_ladder, color='#2ca02c', linewidth=0.8)
axes[1].set_title("[GOOD] Montgomery Ladder - uniform pattern, no information leak")
axes[1].set_xlabel("Sample")
axes[1].set_ylabel("Power")
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.4)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\\n위쪽: 비트 0과 1이 만드는 패턴 길이의 차이를 육안으로 확인 가능")
print("아래쪽: 동일한 패턴의 반복 → 비트 정보가 누설되지 않음")

---
# 정리

| 공격 | 측정 대상 | 비용 (공격자) | 핵심 방어 |
|------|---------|-------------|----------|
| Bellcore (RSA-CRT) | 단 1개의 faulty signature | Voltage/laser glitcher | Sign-then-Verify, redundant CRT |
| Timing | 응답 시간 | 네트워크 RTT 측정 | 상수 시간 비교 (`hmac.compare_digest`) |
| SPA | 전력 1개 trace | Oscilloscope + shunt resistor | Montgomery Ladder, double-and-add-always |

## CC 인증 관점

이 노트북의 세 공격은 **모두 JIL Attack Potential 산정의 핵심 대상**입니다:

- **Bellcore**: AVA_VAN.5 fault injection 카테고리 — *Bespoke equipment*
- **Timing**: AVA_VAN.3 이상 — *Standard equipment* (낮은 비용)
- **SPA**: AVA_VAN.4 이상 — *Specialized equipment*

CC 평가자가 PP-0084 (Security IC) 같은 PP를 적용할 때, 실제로 평가실에서 수행하는 작업의 대부분이 이 세 카테고리에 대한 정량적 저항성 측정입니다. 알고리즘의 수학적 안전성이 아니라, **구현이 얼마나 누설하는가**가 인증의 통과 여부를 결정합니다.

## 추가 실습 아이디어

- DPA 시뮬레이션: 1000개의 random plaintext에 대한 trace에서 AES SubBytes 출력의 Hamming weight 모델로 1-byte 키 복원
- Lucky13 시뮬레이션: TLS CBC의 MAC 검증 시간으로 padding byte 추정
- 마스킹된 AES와 SIFA: 마스킹이 1차 DPA를 막지만 SIFA에는 취약함을 보이기

각 토픽별로 추가 노트북을 만들 수 있습니다.
